# Spelling Correction with the Noisy Channel Model

## Task 1: Data Loading and Preprocessing

First, you need to load the dataset and preprocess it to extract the necessary information. The dataset contains text with misspelled words tagged. Your goal is to create a vocabulary of correct words and a dictionary of error-correction pairs.

Dataset Link: https://titan.dcs.bbk.ac.uk/~roger/holbrook-tagged.dat

In [ ]:
import os
import re
import math
import string
from collections import Counter

import nltk
nltk.download('brown', quiet=True)
nltk.download('words', quiet=True)
nltk.download('gutenberg', quiet=True)
nltk.download('reuters', quiet=True)
from nltk.corpus import brown, words, gutenberg, reuters

# ── 1a. Load Holbrook dataset ──────────────────────────────────────────
data_path = 'holbrook-tagged-1.dat'

if not os.path.exists(data_path):
    try:
        from google.colab import files
        print("Please upload holbrook-tagged-1.dat:")
        uploaded = files.upload()
        data_path = list(uploaded.keys())[0]
    except ImportError:
        pass

with open(data_path, 'r', encoding='latin-1') as f:
    text = f.read()

# Regex to extract (correct_word, wrong_word) pairs
# Errors look like <ERR targ=correct>wrong</ERR>
pattern = r'<ERR targ=([^>]+)>\s*([^<]+?)\s*</ERR>'
matches = re.findall(pattern, text)

# Build error_pairs dict: maps misspelling -> correct spelling
error_pairs = {}
for correct, wrong in matches:
    error_pairs[wrong.lower().strip()] = correct.lower().strip()

# ── 1b. Build a LARGE vocabulary with word frequencies ─────────────────
# Supplement Holbrook words with NLTK corpora (Brown, Gutenberg, Reuters)
# This gives us a realistic English word-frequency distribution.
vocab = Counter()

# Add Holbrook words (both correct and misspelled forms)
for correct, wrong in matches:
    vocab[correct.lower().strip()] += 1
    vocab[wrong.lower().strip()] += 1

clean_text = re.sub(r'<ERR targ=[^>]+>[^<]+</ERR>', '', text)
for w in re.findall(r'[a-zA-Z]+', clean_text):
    vocab[w.lower()] += 1

# Add Brown corpus word frequencies (~1M words of edited English)
for w in brown.words():
    vocab[w.lower()] += 1

# Add Gutenberg corpus (classic literature)
for w in gutenberg.words():
    vocab[w.lower()] += 1

# Add Reuters corpus (news articles)
for w in reuters.words():
    vocab[w.lower()] += 1

# Add NLTK's dictionary words (ensures common English words are present)
for w in words.words():
    if w.lower() not in vocab:
        vocab[w.lower()] = 1  # at least 1 count so they're known

# Filter: keep only alphabetic entries
vocab = Counter({w: c for w, c in vocab.items() if w.isalpha()})

print(f"Total error annotations found: {len(matches)}")
print(f"Error pairs built: {len(error_pairs)}")
print(f"Vocabulary size: {len(vocab):,}")
print(f"Total word tokens: {sum(vocab.values()):,}")
print("\nTop 15 most common words:", vocab.most_common(15))
print("\nExample error pairs:", list(error_pairs.items())[:8])

## Task 2: Building the Noisy Channel Model

The noisy channel model for spelling correction is based on Bayes' theorem. We want to find the most probable correction `c` for a given (potentially misspelled) word `w`:

$$ \text{argmax}_c P(c|w) = \text{argmax}_c \frac{P(w|c)P(c)}{P(w)} $$

Since `P(w)` is the same for all candidate corrections, we can simplify this to:

$$ \text{argmax}_c P(w|c)P(c) $$

Here:
- `P(c)` is the **language model**: the probability of the correction `c` appearing in the text. We can estimate this from the word frequencies in our corpus.
- `P(w|c)` is the **error model**: the probability that the writer produced the word `w` when they intended to write `c`. We'll estimate this based on the edit distance between `w` and `c`.

In [ ]:
TOTAL_WORDS = sum(vocab.values())
VOCAB_SIZE  = len(vocab)

def P_c(c):
    """Language model probability P(c) with Add-1 (Laplace) smoothing.
    Smoothing ensures unseen words get a tiny but non-zero probability
    instead of exactly 0."""
    return (vocab.get(c.lower(), 0) + 1) / (TOTAL_WORDS + VOCAB_SIZE)

def P_w_given_c(w, c):
    """Error model P(w|c) using exponential decay on edit distance.
    This gives MUCH stronger preference to edit-distance-1 candidates
    over edit-distance-2 candidates compared to the simple 1/(d+1) model.
    
    P(w|c) = 0.95^0 = 1.0   if d=0 (exact match)
    P(w|c) = 0.95^1 = 0.95  if d=1 (one edit away)
    P(w|c) = 0.95^4 = 0.81  if d=2 (two edits away)
    """
    dist = edit_distance(w, c)
    return math.pow(0.95, dist * dist)


def noisy_channel_score(w, c):
    """Combine P(w|c) and P(c) per the noisy channel formula."""
    return P_w_given_c(w, c) * P_c(c)

## Task 3: Edit Distance and Candidate Generation

To find potential corrections for a misspelled word, we need to generate a set of 'candidate' words that are close to the original. A common way to do this is to generate all words that are a small 'edit distance' away.

In [ ]:
import numpy as np

# Edit Distance function (Levenshtein distance)
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],    # deletion
                    dp[i][j - 1],    # insertion
                    dp[i - 1][j - 1] # substitution
                )
    return dp[m][n]

def calculate_edit_distance(s1, s2):

    m, n = len(s1), len(s2)

    # Initialize the DP matrix with zeros. Dimensions are (m+1) x (n+1).
    dp = np.zeros((m + 1, n + 1), dtype=int)

    # --- Step 2: Initialization ---
    # Cost of deletions (transforming s1 to empty string)
    for i in range(m + 1):
        dp[i, 0] = i

    # Cost of insertions (transforming empty string to s2)
    for j in range(n + 1):
        dp[0, j] = j

    # --- Step 3: Fill the Matrix ---
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            # If characters are the same, cost is 0, take diagonal value.
            # Note: string indices are i-1 and j-1 because DP table is 1-indexed.
            if s1[i - 1] == s2[j - 1]:
                cost = 0
            else:
                cost = 1 # Cost for substitution

            # Calculate costs for each operation
            deletion_cost = dp[i - 1, j] + 1
            insertion_cost = dp[i, j - 1] + 1
            substitution_cost = dp[i - 1, j - 1] + cost

            # The value of dp[i,j] is the minimum of these three costs
            dp[i, j] = min(deletion_cost, insertion_cost, substitution_cost)

    # --- Step 4: Final Result ---
    edit_distance_val = dp[m, n]

    return edit_distance_val, dp


# ── Fast candidate generation (Norvig-style) ────────────────────────────
# Instead of computing edit distance against every word in the vocab
# (very slow for large vocabularies), we GENERATE all possible strings
# at edit-distance 1 and 2, then check which ones are real words.
# This is O(|alphabet| * |word_length|) instead of O(|vocab|).

LETTERS = 'abcdefghijklmnopqrstuvwxyz'

def _edits1(word):
    """Generate all strings that are 1 edit away from `word`."""
    splits     = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes    = [L + R[1:]            for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces   = [L + c + R[1:]        for L, R in splits if R for c in LETTERS]
    inserts    = [L + c + R             for L, R in splits for c in LETTERS]
    return set(deletes + transposes + replaces + inserts)

def _edits2(word):
    """Generate all strings that are 2 edits away from `word`.
    Only keeps those that are known vocabulary words (for efficiency)."""
    return set(e2 for e1 in _edits1(word) for e2 in _edits1(e1) if e2 in vocab)

def _known(words_set):
    """Filter a set of words to only those present in vocab."""
    return set(w for w in words_set if w in vocab)

def generate_candidates(w, max_dist=2):
    """Generate correction candidates using Norvig's strategy:
    1. If the word itself is known, return it (distance 0).
    2. Else try all edit-distance-1 words that are in vocab.
    3. Else try all edit-distance-2 words that are in vocab.
    4. Fall back to the original word.
    This is orders of magnitude faster than brute-force scanning."""
    candidates = _known([w]) or _known(_edits1(w)) or _edits2(w) or {w}
    return candidates

## Task 4: Create the Spelling Corrector

Now, combine the components to create your spelling corrector. For a given word, you will generate candidates and then score them using the noisy channel model to find the best correction.

In [ ]:
def correct_word(w):
    """Correct a single word using the noisy channel model.
    Handles punctuation: strips trailing/leading punctuation,
    corrects the core word, then re-attaches punctuation."""
    # Strip and save surrounding punctuation
    leading = ''
    trailing = ''
    while w and w[0] in string.punctuation:
        leading += w[0]
        w = w[1:]
    while w and w[-1] in string.punctuation:
        trailing = w[-1] + trailing
        w = w[:-1]

    if not w:  # was all punctuation
        return leading + trailing

    w_lower = w.lower()

    # If w is already a known word with reasonable frequency, return it
    if w_lower in vocab:
        return leading + w_lower + trailing

    # Generate candidates and score them
    candidates = generate_candidates(w_lower)
    if not candidates:
        return leading + w_lower + trailing

    # Pick the candidate that maximizes noisy_channel_score
    best = max(candidates, key=lambda c: noisy_channel_score(w_lower, c))
    return leading + best + trailing


def correct_sentence(sentence):
    """Apply correct_word to every token in the sentence."""
    words = sentence.split()
    corrected = [correct_word(w) for w in words]
    return ' '.join(corrected)


# Test on given sentences
test_sentences = [
    "I have a good freind who is my best companion.",
    "The wether is pleasent today.",
    "My computer is runing sloowly.",
    "Wile I am not sure about this but still I'll eate this."
]

for sent in test_sentences:
    print("Original: ", sent)
    corrected = correct_sentence(sent)
    # Highlight differences
    orig_words = sent.split()
    corr_words = corrected.split()
    highlighted = []
    for o, c in zip(orig_words, corr_words):
        if o.lower() != c:
            highlighted.append(f'[{o} → {c}]')
        else:
            highlighted.append(c)
    print("Corrected:", ' '.join(highlighted))
    print()

## Task 5: Comparison with a Standard Library

Finally, let's compare your model's performance with a well-established spell-checking library, `pyspellchecker`. This will give you a baseline to understand how your model performs and where it could be improved.

In [ ]:
!pip install -q pyspellchecker
from spellchecker import SpellChecker

spell = SpellChecker()

test_sentences = [
    "I have a good freind who is my best companion.",
    "The wether is pleasent today.",
    "My computer is runing sloowly.",
    "Wile I am not sure about this but still I'll eate this."
]

print("=" * 80)
print("COMPARISON: Our Noisy Channel Model  vs  pyspellchecker")
print("=" * 80)

for sent in test_sentences:
    words = sent.split()
    print(f"\n{'─' * 70}")
    print("Original:              ", sent)

    # Our model's correction
    our_correction = correct_sentence(sent)
    print("Our Model:             ", our_correction)

    # Pyspellchecker's correction
    pyspell_words = []
    for w in words:
        # Strip punctuation for pyspellchecker too
        clean = w.strip(string.punctuation)
        if clean:
            correction = spell.correction(clean)
            if correction and correction != clean.lower():
                # Re-attach punctuation
                prefix = w[:len(w) - len(w.lstrip(string.punctuation))]
                suffix = w[len(w.rstrip(string.punctuation)):]
                pyspell_words.append(prefix + correction + suffix)
            else:
                pyspell_words.append(w.lower())
        else:
            pyspell_words.append(w)
    print("Pyspellchecker:        ", ' '.join(pyspell_words))

print(f"\n{'═' * 80}")